> **Public copy.** This step reads the raw Jira export and the test trackers, which stay private, so it cannot be run from this repository. The outputs below are from the original run. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E, releases such as 1.51, System tag 1 to 11 and so on), so every count is exactly as computed.


# 04 Final release for every defect

**Goal:** combine every source into one release per defect, and test how reliable the date match is.

**Vendor first, then release (agreed with Jagan, 23 Nov 2026).** The defect's own words give its vendor: Vendor C words (Vendor C app, Attendant app, Store management app, Handset model, handset, scanner wall, 3.2.x builds) mean Vendor C; Vendor D words (Vendor D, left in trolley, active or silent mode) Vendor D; Vendor E Vendor E; self-checkout words Vendor B; till words Vendor A. When the words name no vendor, Jagan's own component answer is used (Scan and go means Vendor C, Self-checkout Vendor B, Vendor D Vendor D, Vendor E Vendor E).

Order of trust (the first one that applies wins):
0. **Upgrade and patch test work** (Windows 10 upgrade, security patches, WAS, DB or MQ upgrades, server migration, cutover, POC, firmware upgrades, health checks) is not release testing. The trackers keep that effort in its own group (Upgrade/infra, agreed with Jagan), so its defects are kept out of release defect counts too, or defects per person-day would be inflated. Such a defect keeps a release only when its **title** names the release (for example a title that starts "Release 1.42 - ..."). A release in its labels or description is usually just the build that was installed (for example a CI 10 label on an operating-system upgrade test, which the tracker logs separately from CI 10 testing).
1. **Vendor C defects (Vendor C):** if the **title** names the Vendor A or Vendor B release under test, the defect belongs to that release, with Vendor C noted as well, because it was found in that release's testing. A number after System tag 2 is the version of that system being upgraded, not the release under test. Otherwise a Vendor C defect goes to a Vendor C programme: the programme named in the title or labels, else the Vendor C programme whose test window contains the defect's date. Vendor A numbers in labels or fixVersion of a Vendor C defect are not used.
2. **Jira text, high:** the title, labels or fixVersion name a release (notebook 02). Vendor D or Vendor E work inside a Vendor A or Vendor B release stays with that release, with Vendor D or E noted as well.
3. **Tracker key:** the defect's Jira key is written in a tracker row for a release (notebook 03).
4. **Jira text, medium or low:** environment, description, then comments, only if it agrees with the vendor words.
5. **Vendor D and Vendor E defects:** a programme named in the text, else the date inside their own programme windows, else the date inside the Vendor A (and for Vendor E, Vendor B) windows they are delivered into.
6. **Date only, inside the vendor's windows:** till words use Vendor A windows, self-checkout words Vendor B windows; with no vendor words, Vendor A or Vendor B windows. Exactly one window must contain the date; overlapping windows are a tie and stay unassigned (ties or low scores stay unassigned).
7. Otherwise: **unassigned**.

Before trusting the date match, we test it: we hide the release of defects we already know, run the date match, and count how often it is right.

In [1]:
import sqlite3, sys
from pathlib import Path
import pandas as pd

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
WORK = ROOT / 'data' / 'private' / 'hindsight_work.sqlite'    # Jira export (read only)
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
print('Reading from:', WORK)
import numpy as np
from hs_vendor import vendor_from_text
from hs_programmes import programme_named
from hs_trackers import VENDOR_OF_STREAM
with sqlite3.connect(CLEAN) as cc:
    d = pd.read_sql('select key, summary, created, labels, environment, counts_for_risk from defects', cc)
    rel = pd.read_sql('select * from defect_release', cc)
    runs = pd.read_sql('select * from release_window_runs', cc, parse_dates=['run_start', 'run_end'])
    windows = pd.read_sql('select * from release_windows', cc)
    tkeys = pd.read_sql("select * from tracker_defect_keys where work_group = 'release'", cc)
d['created'] = pd.to_datetime(d.created).dt.tz_localize(None)
x = d.merge(rel, on='key', how='left')
x['vendor_hint'] = [vendor_from_text(a, b) for a, b in zip(x.summary, x.labels)]
x['vendor_source'] = np.where(x.vendor_hint.notna(), 'words', None)

# No vendor words: use Jagan's hand-given main component, where it points to one vendor
COMP_VENDOR = {'Scan and go (handsets)': 'Vendor C', 'Self-checkout (SCO)': 'Vendor B', 'Vendor D': 'Vendor D', 'Vendor E': 'Vendor E'}
ovr = pd.read_csv(ROOT / 'data' / 'private' / 'component_overrides.csv')
comp_vendor = ovr.set_index('key')['main'].map(COMP_VENDOR).dropna()
fill = x.vendor_hint.isna() & x.key.isin(comp_vendor.index)
x.loc[fill, 'vendor_hint'] = x.loc[fill, 'key'].map(comp_vendor)
x.loc[fill, 'vendor_source'] = "Jagan's component"
print(int(fill.sum()), "defects get their vendor from Jagan's component answer")
x.vendor_hint.value_counts(dropna=False).to_frame('defects')

Reading from: data/private/hindsight_work.sqlite
11 defects get their vendor from Jagan's component answer


,defects
vendor_hint,
NaN,807
Vendor C,784
Vendor B,750
Vendor A,676
Vendor D,82
Vendor E,53


## 0. Which Jira field shows the release a defect was found in?

Notebook 02 reads the title first, then labels, then fixVersion. The check below supports that order: for defects where two fields name different releases, we look at which release's tracker test window contains the date the defect was raised.

In [2]:
with sqlite3.connect(CLEAN) as cc:
    mentions = pd.read_sql('select key, source, main from release_mentions', cc)
created = d.set_index('key').created

def first_named(source):
    return mentions[mentions.source == source].drop_duplicates('key').set_index('key').main

def in_window(key, release):
    r = runs[runs.release_key == release]
    if r.empty:
        return 'no window'
    c = created[key]
    return 'date inside' if ((r.run_start <= c) & (c <= r.run_end)).any() else 'date outside'

title = first_named('summary')
for other in ['fixVersion', 'label']:
    pair = pd.concat([title, first_named(other)], axis=1, join='inner', keys=['title', other])
    pair = pair[pair['title'] != pair[other]]
    table = pd.crosstab([in_window(k, v) for k, v in pair['title'].items()],
                        [in_window(k, v) for k, v in pair[other].items()],
                        rownames=['title release'], colnames=[other + ' release'])
    print(f'Title and {other} name different releases for {len(pair)} defects:')
    print(table, end='\n\n')

Title and fixVersion name different releases for 46 defects:
fixVersion release  date inside  date outside  no window
title release                                           
date inside                   6            21          0
date outside                  2             2          0
no window                     0             0         15

Title and label name different releases for 54 defects:
label release  date inside  date outside  no window
title release                                      
date inside              3            29          2
date outside            11             8          0
no window                1             0          0



**Reading this:** when the title and fixVersion disagree, the title's release usually contains the date the defect was raised and fixVersion's does not, because fixVersion is often the later release that carried the fix. Labels behave the same way, less strongly. So the title is the best record of the release a defect was **found** in.

## 1. The date match

The window is the exact test window from the trackers (no extra days before or after). Settings with extra days, and a "latest release wins" tie-break, were tried and were less accurate, so they are not used.
`streams` is the list of release streams to try, in order (for example Vendor D first, then Vendor A).

In [3]:
def date_match(created, streams):
    for st in streams:
        m = runs[(runs.release_stream == st) & (runs.run_start <= created) & (created <= runs.run_end)]
        if m.release_key.nunique() == 1:
            return m.release_key.iloc[0], st, 'unique'
        if m.release_key.nunique() > 1:
            return None, None, f'tie ({m.release_key.nunique()} releases)'
    return None, None, 'no window'

STREAMS_FOR = {'Vendor A': ['Vendor A'], 'Vendor B': ['Vendor B'], 'Vendor C': ['Vendor C'], 'Vendor D': ['Vendor D', 'Vendor A'],
               'Vendor E': ['Vendor E', 'Vendor A', 'Vendor B'], None: ['Vendor A', 'Vendor B']}
def streams_for(v):
    return STREAMS_FOR.get(v if isinstance(v, str) else None)

## 2. Test it on defects whose release we already know

We take defects with a high-confidence Vendor A or Vendor B release from Jira text, whose release has a tracker window, hide the answer, and run the date match with the same vendor rule.

In [4]:
known = x[x.confidence.eq('high') & x.release_main.isin(set(runs.release_key)) & (x.vendor_hint != 'Vendor C')].copy()
known[['pred', 'pred_stream', 'how']] = [date_match(c, streams_for(h)) for c, h in zip(known.created, known.vendor_hint)]
known['result'] = np.where(known.pred.isna(), 'not matched', np.where(known.pred == known.release_main, 'right', 'wrong'))
print(len(known), 'known defects tested')
print(known.result.value_counts().to_string())
hit = known[known.pred.notna()]
print(f"\nWhen the date match gives an answer, it is right {(hit.pred == hit.release_main).mean():.0%} of the time "
      f"({len(hit)} defects). It gives an answer for {known.pred.notna().mean():.0%} of them.")

1149 known defects tested
result
right          750
not matched    239
wrong          160

When the date match gives an answer, it is right 82% of the time (910 defects). It gives an answer for 79% of them.


In [5]:
known['year'] = known.created.dt.year
known.groupby('year').apply(lambda g: pd.Series({
    'tested': len(g),
    'answered %': round(100 * g.pred.notna().mean()),
    'right % of answered': round(100 * (g[g.pred.notna()].pred == g[g.pred.notna()].release_main).mean()) if g.pred.notna().any() else None,
}), include_groups=False)

,tested,answered %,right % of answered
year,,,
2017,1.0,0.0,NaN
2018,26.0,12.0,100.0
2019,246.0,65.0,85.0
2020,132.0,85.0,86.0
2021,119.0,81.0,83.0
2022,70.0,93.0,69.0
2023,72.0,97.0,89.0
2024,201.0,92.0,89.0
2025,203.0,87.0,72.0


**Reading this:** the date match is right most of the time but not always, so date-only releases keep their own confidence label. The model is run twice: on text-confirmed releases only, and including date-only matches. If the results agree, the date match is not distorting them.

## 3. Final release for every defect

In [6]:
tk = tkeys.drop_duplicates('key')[['key', 'release_key', 'release_stream']].rename(
    columns={'release_key': 'tk_release', 'release_stream': 'tk_stream'})
x = x.merge(tk, on='key', how='left')

def agrees(vendor_hint, stream):
    # a weaker text match is kept only if it agrees with the defect's vendor words
    if vendor_hint in ('Vendor C', 'Vendor E'):
        return False
    return not isinstance(vendor_hint, str) or VENDOR_OF_STREAM.get(stream) == vendor_hint or vendor_hint == 'Vendor D'

import re
SYSTEM_TAG_2_VERSION = re.compile(r'system tag 2\s*[-_:]?\s*v?\s*1\.\d\d', re.I)   # a version number right after System tag 2 is that system's version, not the release
UPGRADE_WORK = re.compile(r'windows ?1[01]|\bwin ?1[01]\b|security ?patch|patching|\bwas\b.{0,12}(patch|upgrade)|was ?upgrade|'
                          r'database ?db ?upgrade|\bdb ?upgrade|mq ?upgrade|mcafee ?upgrade|server ?migration|cutover|\bpoc\b|'
                          r'firmware ?upgrade|\bos ?upgrade|motherboard|health.?check|upgrade ?test', re.I)
x['upgrade_work'] = [bool(UPGRADE_WORK.search(f'{a} || {b}')) for a, b in zip(x.summary, x.labels.fillna(''))]

final = []
for r in x.itertuples():
    v = r.vendor_hint if isinstance(r.vendor_hint, str) else None
    if r.upgrade_work and r.release_source != 'summary':
        final.append((None, None, 'upgrade / infra work (no release)', 'upgrade work'))
    elif v == 'Vendor C':
        k = programme_named('Vendor C', r.summary, r.labels)
        if r.release_source == 'summary' and not SYSTEM_TAG_2_VERSION.search(r.summary):
            final.append((r.release_main, r.stream, 'Jira text (title names the release under test)', 'high'))
        elif k:
            final.append((k, 'Vendor C', 'Vendor C programme named in text', 'high'))
        else:
            rk, st, how = date_match(r.created, ['Vendor C'])
            final.append((rk, st, 'vendor then date', 'date only') if rk else (None, None, 'vendor only: ' + how, 'vendor only'))
    elif r.confidence == 'high':
        final.append((r.release_main, r.stream, 'Jira text', 'high'))
    elif isinstance(r.tk_release, str):
        final.append((r.tk_release, r.tk_stream, 'tracker key', 'high'))
    elif r.confidence in ('medium', 'low') and agrees(v, r.stream):
        final.append((r.release_main, r.stream, 'Jira text (' + r.release_source + ')', r.confidence))
    elif v in ('Vendor D', 'Vendor E'):
        k = programme_named(v, r.summary, r.labels)
        if k:
            final.append((k, 'Vendor D' if v == 'Vendor D' else 'Vendor E', 'programme named in text', 'high'))
        else:
            rk, st, how = date_match(r.created, streams_for(v))
            final.append((rk, st, 'vendor then date', 'date only') if rk else (None, None, 'vendor only: ' + how, 'vendor only'))
    else:
        rk, st, how = date_match(r.created, streams_for(v))
        if rk:
            final.append((rk, st, 'vendor then date' if v else 'date only', 'date only'))
        else:
            final.append((None, None, 'unassigned: ' + how, 'none'))
x[['release_key', 'release_stream', 'method', 'final_confidence']] = final
x['vendor'] = x.release_stream.map(VENDOR_OF_STREAM)
no_rel = x.release_key.isna() & x.vendor_hint.notna()
x.loc[no_rel, 'vendor'] = x.loc[no_rel, 'vendor_hint']
# Vendor C, Vendor D or Vendor E work found inside a Vendor A or Vendor B release involves Vendor C, D or E as well
x['also_vendor'] = np.where(x.vendor_hint.isin(['Vendor C', 'Vendor D', 'Vendor E']) & (x.vendor != x.vendor_hint), x.vendor_hint, None)
x.final_confidence.value_counts().to_frame('defects')

,defects
final_confidence,
high,1825
date only,537
none,373
upgrade work,135
vendor only,134
medium,88
low,60


In [7]:
x.method.str.replace(r': tie.*', ': tie', regex=True).value_counts().to_frame('defects')

,defects
method,
Jira text,1358
vendor then date,396
Vendor C programme named in text,388
unassigned: no window,321
date only,141
upgrade / infra work (no release),135
vendor only: no window,129
Jira text (comment),60
Jira text (title names the release under test),56


In [8]:
print(f"Release known for {x.release_key.notna().mean():.0%} of all defects")
print(f"  with text or tracker evidence: {x.final_confidence.isin(['high','medium','low']).mean():.0%}")
print(f"  by date only: {x.final_confidence.eq('date only').mean():.0%}")
risk = x[x.counts_for_risk == 1]
print(f"Among the {len(risk)} defects that count for risk: {risk.release_key.notna().mean():.0%} have a release")
summary = x.groupby('vendor', dropna=False).agg(defects=('key', 'size'), with_release=('release_key', lambda s: int(s.notna().sum())))
summary['also involved (Vendor C / Vendor D / Vendor E inside a Vendor A or Vendor B release)'] = x.groupby('also_vendor').size()
summary

Release known for 80% of all defects
  with text or tracker evidence: 63%
  by date only: 17%
Among the 2597 defects that count for risk: 79% have a release


,defects,with_release,also involved (Vendor C / Vendor D / Vendor E inside a Vendor A or Vendor B release)
vendor,,,
Vendor A,1614,1462,NaN
Vendor B,650,444,NaN
Vendor C,728,586,56.0
Vendor D,1,1,81.0
Vendor E,23,17,30.0
NaN,136,0,NaN


In [9]:
per = x[x.release_key.notna()].groupby(['vendor', 'release_key', 'final_confidence']).size().unstack(fill_value=0)
per['total'] = per.sum(axis=1)
per

final_confidence                       date only  high  low  medium  total
vendor   release_key                                                      
Vendor A 1.28                                  0     0    0       1      1
         1.29                                  0    10    0       0     10
         1.30                                  0     1    0       1      2
         1.31                                  0     3    0       2      5
         1.32                                  0     5    1       1      7
         1.33                                  0    74    0       0     74
         1.34                                  0    84    0      13     97
         1.35                                  0     1    0       0      1
         1.36                                  0    52    4      17     73
         1.37                                  2    71    5      15     93
         1.39                                 22   199   20       7    248
         1.41                                 18    61    5       0     84
         1.42                                  1    57    3       1     62
         1.43                                  7    49    0       5     61
         1.44                                  4    32    0       0     36
         1.45                                  6    32    1       1     40
         1.46                                 17    18    0       0     35
         1.47                                 29    27    2       5     63
         1.48                                 51    30    1       3     85
         1.49                                 18    35    0       0     53
         1.50                                 34    52    2       5     93
         1.51                                  5     7    0       2     14
         1.52                                 15    70    2       1     88
         1.53                                 13    22    0       4     39
         1.54                                 17    32    0       1     50
         1.55                                 13     0    0       0     13
         1.56                                  6    28    1       0     35
Vendor B CI 1                                  0     1    0       0      1
         CI 10                                21   107    0       0    128
         CI 11                                 4    39    0       0     43
         CI 11.1                               1     8    0       0      9
         CI 11.2                               0    12    0       0     12
         CI 11.3                               2     3    0       0      5
         CI 11.4                               0     2    0       0      2
         CI 2                                  1    22    0       1     24
         CI 3                                  7    28   10       0     45
         CI 4                                  0    46    1       0     47
         CI 5                                  1    21    0       0     22
         CI 6                                  1    21    2       1     25
         CI 7                                  3    19    0       1     23
         CI 8                                 10    18    0       0     28
         CI 9                                  1    29    0       0     30
Vendor C 5.1                   2     1    0       0      3
         Vendor C 5.2         15    20    0       0     35
         Vendor C 5.0                         11   224    0       0    235
         Vendor C 5.4                      0    47    0       0     47
         Vendor C 4.4                      18     0    0       0     18
         Vendor C 3.0           13    67    0       0     80
         Vendor C 4.3                  74     0    0       0     74
         Vendor C 4.0                     6    17    0       0     23
         Vendor C 4.2                        33    10    0       0     43
         Vendor C 4.1                    25     0    0       0     25
         Vendor C 4.5

## 4. Save

In [10]:
x['work_type'] = np.where(x.final_confidence == 'upgrade work', 'upgrade / infra work', 'release testing')
out = x[['key', 'release_key', 'release_stream', 'vendor', 'also_vendor', 'vendor_hint', 'vendor_source', 'work_type', 'method', 'final_confidence',
         'n_releases_named', 'releases_named', 'inc', 'ptf', 'build_label']]
val = known[['key', 'release_main', 'pred', 'how', 'result', 'year']]
with sqlite3.connect(CLEAN) as cc:
    out.to_sql('defect_release_final', cc, if_exists='replace', index=False)
    val.to_sql('date_match_validation', cc, if_exists='replace', index=False)
print('saved', len(out))

saved 3152
